# SINDYc 4 · Letting SINDy choose the switches, and transfer to another unit type (exploratory)

Two changes to the model of `sindyc_02_world_model.ipynb`:

1. **SINDy chooses the switches.** The reach closure and the movement gates were fitted by hand
   with `curve_fit` (about 30 parameters) before SINDy ran. Here each mechanism gets a family of
   candidate switches at different distances, and sparse selection keeps one. The geometry inside
   each switch has no fitted parameters.
2. **Dimensionless form and a second unit type.** Distances in units of reach, time in units of
   how long one unit takes to kill another, speeds in units of walking speed. If the coefficients
   are physical, a model fitted on marines should predict **stalker** battles without refitting.

Code: `models/switch_library.py`. Data: exp06 (marines, 6,000 + 1,000 battles) and **exp07**
(stalkers, 1,500 + 300 battles, the same intents and sampler; `scripts/collect_stalkers.py`).
Both exploratory (`allow_dirty`).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import nnls

from sindy_wm.models.intent_model import (
    DT,
    IntentModel,
    add_controls,
    add_rates,
    fit_closures,
    horizon_errors,
    make_batch,
    r2,
)
from sindy_wm.models.switch_library import (
    UNITS,
    choose_switches,
    dimensionless,
    fire_profile,
    fire_switch,
    scaled,
)
from sindy_wm.paths import DATA_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)


def load(experiment, name):
    d = add_rates(add_controls(pd.read_parquet(DATA_DIR / "processed" / experiment / f"{name}.parquet")))
    return d, d[(d["alive_blue"] > 0) & (d["alive_red"] > 0)]


m_val, m_val_rows = load("exp06_intents", "val")
_, m_rows = load("exp06_intents", "train")
s_val, s_val_rows = load("exp07_stalkers", "val")
_, s_rows = load("exp07_stalkers", "train")
pd.DataFrame(UNITS).T.round(2)

`tau` is health per unit over the health a unit loses per second to one shooter: 4.6 s for
marines, 12.3 s for stalkers (shields and armor included: 18 damage per hit on shields, 17 on hp).

## 1 · Can plain SINDy pick the reach switch?

Candidates for the losses Blue inflicts: `S(c)` = Blue units within c of Red's front, for c on a
grid of 4-8 in steps of 0.1, each also multiplied by withdraw. Plus alive and health as
alternatives. First plain STLSQ, then STLSQ with the physical sign (firing can only remove
health, withdrawing can only reduce fire).

In [ ]:
grid = np.round(np.arange(4.0, 8.01, 0.1), 2)
sub = m_rows.sample(100_000, random_state=0)


def candidates(d):
    cols, names = [], []
    for c in grid:
        s = fire_switch(d, "blue", c)
        cols += [s[:, 0], -s[:, 1]]
        names += [f"S({c})", f"-S({c})*wd"]
    cols += [d["alive_blue"].to_numpy(), d["health_blue"].to_numpy()]
    return np.column_stack(cols), names + ["alive", "health"]


def stlsq_signed(T, y, threshold, signed):
    scale = T.std(0) + 1e-12
    Ts, keep = T / scale, np.ones(T.shape[1], bool)
    for _ in range(20):
        c = np.zeros(T.shape[1])
        c[keep] = (
            nnls(Ts[:, keep], y, maxiter=5000)[0]
            if signed
            else np.linalg.lstsq(Ts[:, keep], y, rcond=None)[0]
        )
        new = np.abs(c) >= threshold * y.std()
        if (new == keep).all():
            break
        keep = new
    return c / scale


T, names = candidates(sub)
Tv, _ = candidates(m_val_rows)
y, yv = -sub["d_health_red"].to_numpy(), -m_val_rows["d_health_red"].to_numpy()
for signed in (False, True):
    c = stlsq_signed(T, y, 0.1, signed)
    kept = {n: round(v, 3) for n, v in zip(names, c, strict=True) if v != 0}
    print(
        f"{'sign-constrained' if signed else 'plain'} STLSQ: {len(kept)} terms, R² val {r2(yv, Tv @ c):.3f}"
    )
    print("   ", kept)

**What the numbers say:** plain STLSQ keeps many neighbouring switches with large coefficients
of opposite sign: the candidates are nearly collinear and their differences fit noise. With the
physical sign, only a few switches survive, all between about 5.8 and 7. So the library can
carry the switch, but the selection needs a constraint: either signs, or **one switch per
mechanism** (best subset of size one, as in MIOSR-type SINDy). The rest uses the second.

## 2 · One switch per mechanism, chosen by fit

For fire: the R² of the losses each side inflicts, with one switch at each candidate distance.
For movement: the same for each movement's closing switch (distance and width), one mechanism
at a time. Chosen on training battles.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for side in ("blue", "red"):
    p = fire_profile(m_rows.sample(100_000, random_state=1), side)
    ax.plot(p.index, p.values, label=f"losses {side} inflicts")
ax.axvline(UNITS["marine"]["reach"], color="k", ls=":", lw=1)
ax.set(xlabel="switch distance c (from the enemy front)", ylabel="R² (train)", ylim=(0, 0.8), title="marines")
ax.legend()

P_marine, chosen = choose_switches(m_rows)
pd.Series(chosen).round(2)

Then the usual SINDYc fit (`IntentModel`, STLSQ threshold 0.2) on the chosen switches, against
the hand-fitted closures of notebook 2 (also fitted on the closure, not on measured reach).

In [ ]:
P_hand = fit_closures(m_rows)
marine = {
    "hand-fitted closures": IntentModel(P_hand).fit(m_rows, thr_health=0.2, thr_shape=None),
    "SINDy-chosen switches": IntentModel(P_marine).fit(m_rows, thr_health=0.2, thr_shape=None),
}
for name, m in marine.items():
    print(f"== {name}")
    m.print()
Bm = make_batch(m_val)
pd.DataFrame(
    {k: horizon_errors(m.simulate(Bm["X0"], Bm["shape0"], Bm["U"]), Bm) for k, m in marine.items()}
).T.round(3)

**What the numbers say:** fill in. Compare the health columns (does replacing the closure fit
cost accuracy?) and the distance columns separately, and count the parameters fitted outside
SINDy: hand closures 2 x 6 + 3 x 8 + 4; switches 6 distances + 4 widths, each picked from a grid.

## 3 · Are the coefficients the same in unit-free form?

The same two procedures, run separately on stalker battles. Then every coefficient in
dimensionless form: firepower x tau, closing speeds / walking speed, switch distances / reach.

In [ ]:
P_stalker, chosen_s = choose_switches(s_rows)
stalker = {
    "hand-fitted closures": IntentModel(fit_closures(s_rows)).fit(s_rows, thr_health=0.2, thr_shape=None),
    "SINDy-chosen switches": IntentModel(P_stalker).fit(s_rows, thr_health=0.2, thr_shape=None),
}
for name in marine:
    print(f"== {name}")
    display(
        pd.DataFrame(
            {
                "marine": dimensionless(marine[name], "marine"),
                "stalker": dimensionless(stalker[name], "stalker"),
            }
        ).round(3)
    )

**What the numbers say:** fill in. Which coefficients are the same for both unit types, which
differ, and is there a physical reason for each difference?

## 4 · Transfer: predict stalker battles with the marine model, no refit

Validation battles of exp07 (stalkers), start points every 1 s, real intents applied.

- **as is:** the marine model's coefficients unchanged
- **scaled:** rewritten with stalker reach, tau and speed (`scaled`), nothing fitted on stalkers
- **refit:** fitted on stalker training battles (the best this form can do)

In [ ]:
Bs = make_batch(s_val)
steps = Bs["U"].shape[1]
transfer = {"persistence": horizon_errors(np.repeat(Bs["X0"][:, None], steps + 1, 1), Bs)}
for name in marine:
    transfer[f"{name}: marine, as is"] = horizon_errors(
        marine[name].simulate(Bs["X0"], Bs["shape0"], Bs["U"]), Bs
    )
    m = scaled(marine[name], "marine", "stalker")
    transfer[f"{name}: marine, scaled"] = horizon_errors(m.simulate(Bs["X0"], Bs["shape0"], Bs["U"]), Bs)
    transfer[f"{name}: refit on stalkers"] = horizon_errors(
        stalker[name].simulate(Bs["X0"], Bs["shape0"], Bs["U"]), Bs
    )
pd.DataFrame(transfer).T.round(3)

Horizons are in seconds. Stalker battles are slower (tau 12.3 s against 4.6 s), so 4 s is a
smaller part of a stalker battle: compare rows within this table, not with the marine table.

In [ ]:
gap = pd.DataFrame(transfer).T
for name in marine:
    as_is, sc, ref = (
        gap.loc[f"{name}: {k}", "health 4 s"]
        for k in ("marine, as is", "marine, scaled", "refit on stalkers")
    )
    print(
        f"{name}: scaling closes {(as_is - sc) / (as_is - ref):.0%} of the gap",
        "between 'as is' and 'refit' (health, 4 s)",
    )
print(f"one step is {DT} s; stalker battles last {s_val.groupby(level=0)['t'].max().mean():.0f} s on average")

## Read with care

- One second unit type, same map, same intents, same Red AI. Stalkers differ from marines in
  reach, speed, health, shields and armor at once: the test can't say which scaling carries the
  transfer.
- The switch grids and the width grid are choices; best subset over a grid is a brute-force
  version of what MIOSR does with an optimiser.
- The alive exponent p is not scaled: shields change how health loss turns into deaths.
- Exploratory data, models chosen while looking at validation numbers.

## Questions for you

1. The fire switch lands above the reach (c / reach about 1.1 for marines). Is that a flaw of
   the switch geometry, or a real effect (units walk into range before they fire)?
2. Which differences between the marine and stalker coefficients would you expect from the game's
   rules, and which surprise you?
3. Does "scaled marine model beats refit-free baselines on stalkers" answer RQ4, or only part of
   it? What would a convincing second test be?